# 3 — Routing tables (C) and the full pipeline, headless

This notebook completes the steering stack and then runs the **entire pipeline without
the chat UI**: the same recipe → agent → structured answer path `app.py` drives, from
plain Python. Reference: [`docs/RETRIEVAL.md`](../RETRIEVAL.md) §7 and
[`docs/RECIPES.md`](../RECIPES.md).

**Prerequisites:** the same as notebook 02 (Neo4j + backfilled categories + embed
model), plus — for the agent section only — `ANTHROPIC_API_KEY` in the env file
(`models.yaml` binds the agent role to Claude). Sections 1–2 run without any LLM.

In [ ]:
# Environment: credentials live in ema_nlp.env (never in the repo). config.py
# searches: $EMA_ENV_FILE -> ~/Nextcloud/Datasets/ema_nlp/ema_nlp.env -> ~/.myenvs/.
# On this host the project Neo4j container runs on alt ports; if your env file
# does not set NEO4J_URI, the setdefault below points at the project container.
import os
import sys
from pathlib import Path

# Make `harness` importable when the notebook is started from docs/examples/.
REPO = Path.cwd()
while not (REPO / "harness").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import config  # noqa: E402,F401  (loads ema_nlp.env)

# After the env file: only fill NEO4J_URI if it set neither.
os.environ.setdefault("NEO4J_URI", "bolt://localhost:7688")  # project container (alt port)
print("repo root:", REPO)

## 1. Option C — the routing table

A routing table maps query signals to a **category prior** — the "if you ask about X,
look in guidelines first" knowledge — kept entirely as data
(`harness/configs/routing/default.yaml`; drop a same-named file under
`$EMA_CONFIG_DIR/routing/` to shadow it). Rules are ordered, first-match-wins,
word-boundary, case-insensitive. Each rule carries a `mode`:

- **`prefer`** (default, soft) — results are *reordered* with the routed categories
  first; nothing is excluded.
- **`filter`** (hard) — retrieval is *restricted*; if that yields nothing, `ema_search`
  automatically retries unfiltered and says so.

The router itself is generic code; the topics live in the YAML.

In [ ]:
from harness.retrieval import load_router

router = load_router("default")
print(f"table 'default': {len(router.rules)} rules\n")

for q in [
    "What is the acceptable intake for NDMA impurities?",
    "Which variations require a worksharing procedure?",
    "Summarise the Keytruda assessment report conclusions.",
    "What is the capital of France?",   # no regulatory signal -> no prior
]:
    print(f"{q}\n  -> {router.route(q)}\n")

### Building a custom table

For experiments you can construct a router in memory; for real use, write a YAML and
load it by name (a recipe selects it with `retrieval.routing: <name>`).

In [ ]:
from harness.retrieval import QueryRouter, RoutingRule

custom_router = QueryRouter([
    RoutingRule(
        name="my_pharmacovigilance",
        keywords=("signal detection", "PSUR", "safety update"),
        categories=("qa", "scientific_guideline"),
        mode="prefer",
    ),
])
print(custom_router.route("When is a PSUR submission due?"))

## 2. The `ema_search` tool, standalone (no LLM)

The agent's search tool can be built and called directly — useful to see exactly what
the agent sees. Note in the output:

- the bracketed **steering note** (`[routing: ...]` / `[category filter: ...]`) — every
  applied steering step is visible to the agent and on the trace;
- the **`category=`** tag on every result;
- **`via=link_expansion`** on link-expanded results.

Precedence is strict: an explicit `source_category` argument **beats** the router; the
router only supplies the default prior.

In [ ]:
from harness.indexing import build_retriever, load_index_profile, open_index
from harness.tools import get_tool

profile = load_index_profile("neo4j_steered")
index = open_index(profile)
retriever = build_retriever(profile, index)

tool = get_tool("ema_search", retriever=retriever, router=router)

# Routed: the 'impurity_toxicology_limits' rule applies its prefer-prior.
print(str(tool.call(query="What is the acceptable intake for NDMA impurities?"))[:2000])

In [ ]:
# Explicit agent intent wins over the router (hard filter, honest note):
print(str(tool.call(
    query="What is the acceptable intake for NDMA impurities?",
    source_category="epar",
))[:1200])

## 3. The full pipeline — what the chat UI runs, without the chat UI

There is one engine: a recipe (`harness/configs/recipes/*.yaml`) becomes a LlamaIndex
`FunctionAgent` wrapped in an `AgentWorkflowAdapter`. The `steered_agent` recipe wires
all three options at once:

```yaml
retrieval:
  index_profile: neo4j_steered   # A: quotas + oversample, B: LINKS_TO expansion
  pipeline: none
  routing: default               # C: the routing table from section 1
```

plus the `agent_regulatory.md` prompt, which teaches the agent to read the `category=`
tags and steer follow-up searches with `source_category`. `build_recipe` is the single
composition path — the UI, the CLI, and the eval runner all go through it.

*(This cell calls the Anthropic API — it needs `ANTHROPIC_API_KEY` and costs one agent
run.)*

In [ ]:
from harness.recipes import build_recipe, get_recipe

recipe = get_recipe("steered_agent")
pipeline = build_recipe(recipe, index)   # reuses the index opened above

result = await pipeline.ainvoke(
    {"question": "What is the acceptable intake for NDMA impurities?"}
)
print(result["answer_text"])

### Inspect the structured result

The adapter returns the full contract the UI consumes: the typed `RegulatoryAnswer`
(verbatim claim spans + citations), the numbered references with full source passages,
and the claim-span attribution.

In [ ]:
answer = result["answer"]                       # RegulatoryAnswer (pydantic)
print("confidence:", answer.confidence)
print("claims:", len(answer.claims), "| citations:", len(answer.citations), "\n")

for ref in result["references"]:            # numbered by first use in the answer
    print(f"[{ref['n']}] {ref['category'] or '?':22s} {ref['source_url']}")

### Honest stamping

The *resolved* recipe — including `ema.retrieval.routing` — is stamped on every MLflow
trace, so a result can always be tied back to the exact steering configuration that
produced it. To record traces headless, point MLflow at the project store before
invoking (`scripts/run_ui.sh` serves the same store at `http://localhost:5000`):

In [ ]:
import json

print(json.dumps(recipe.resolved_attributes(), indent=2, default=str))

# Optional — record headless runs/traces to the project MLflow store:
# from harness.obs import setup_mlflow
# setup_mlflow("steering-demo")   # then re-run pipeline.ainvoke(...)

## Where to go from here

- **Measure before tuning** (the project's scope lock): run the benchmark with and
  without steering and compare per-question-type —
  `python scripts/run_eval.py --recipe regulatory_agent` vs
  `python scripts/run_eval.py --recipe steered_agent`.
- **Tune from evidence:** SME "prefer *category*" citation verdicts
  (`preferred_category` MLflow assessments, see [`docs/CITATIONS.md`](../CITATIONS.md))
  are the intended signal for adjusting quotas, routing rules, and
  `doc_type_priority` orders.
- **Extend:** new routing rules and category-vocabulary entries are data changes
  (`configs/routing/*.yaml`, `doc_categories.py` rules + a re-run of the backfill).